# DPA-KT — Kết quả 200 epoch + Cross-Validation 5-fold

Notebook này tổng hợp kết quả của **đợt huấn luyện 200 epoch kèm
cross-validation 5-fold** trên tất cả các dataset của benchmark DPA-KT.
Đợt chạy được điều khiển bởi `scripts/train_200_cv.py`; mỗi fold ghi
checkpoint, `log.csv` theo từng epoch và `test_metrics.json` vào thư mục
`runs-200-epochs/<dataset>_full_fold<i>/`.

Nghiên cứu ablation 50 epoch ban đầu (các kết quả theo từng ablation được
tái sử dụng bên dưới) nằm trong `runs-50-epochs/` và notebook tổng
`DPA_KT_master.ipynb`; notebook này tập trung vào đợt huấn luyện dài hơn
có kiểm chứng CV và được chia thành:

1. Môi trường & thiết lập
2. Các dataset trong đợt chạy
2b. Các module mô hình DPA-KT — giải thích cho người mới
2c. Mô tả chi tiết các dataset
2d. Đồ thị KC (đồ thị nút)
3. Kết quả 5-fold CV — tóm tắt theo dataset (mean ± std qua các fold)
4. So sánh metric test với tài liệu pyKT
5. Đường cong huấn luyện từng fold
6. Throughput & bộ nhớ đỉnh
7. Kết luận


## 1. Môi trường & thiết lập

In ra trạng thái torch / CUDA / GPU. Đợt CV 5-fold được chạy trên một GPU NVIDIA GB10 (Grace-Blackwell, bộ nhớ hợp nhất) được chia sẻ.

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt

RUNS_200 = Path('../runs-200-epochs')
RUNS_50  = Path('../runs-50-epochs')
assert RUNS_200.exists(), RUNS_200
print('runs-200-epochs exists:', RUNS_200.exists())
print('runs-50-epochs  exists:', RUNS_50.exists())
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    f,t = torch.cuda.mem_get_info()
    print(f'GPU {torch.cuda.get_device_name(0)} | free {f/1e9:.1f} / {t/1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


Trạng thái torch / CUDA và đường dẫn tới `runs-200-epochs/` (mọi fold của đợt chạy đều nằm ở đây).

## 2. Các dataset trong đợt chạy

Đợt chạy bao gồm **bảy cấu hình dataset** giống như notebook tổng. Mỗi dataset có 5 fold (5-fold CV trên tập train+valid 80% sinh viên, cộng thêm tập test 20% giữ riêng). Với mỗi (dataset, fold) ta huấn luyện **mô hình đầy đủ** tối đa 200 epoch với early stopping (patience = 10) và lưu `best.pt`, `last.pt`, `log.csv`, `test_metrics.json`.

In [ ]:
DATASETS = ['assist09','algebra05','bridge06','xes3g5m','assist12','eedi','junyi']
FOLDS = [0,1,2,3,4]

def run_dir(ds, fold):
    return RUNS_200 / f'{ds}_full_fold{fold}'

# status per (dataset, fold)
status = []
for ds in DATASETS:
    for f in FOLDS:
        r = run_dir(ds, f)
        tm = r / 'test_metrics.json'
        lc = r / 'log.csv'
        if tm.exists():
            s = 'DONE'
        elif lc.exists():
            s = 'PARTIAL'
        else:
            s = 'PENDING'
        status.append({'dataset': ds, 'fold': f, 'status': s})
df_status = pd.DataFrame(status)
pivot = df_status.pivot(index='dataset', columns='fold', values='status')
pivot = pivot.reindex(DATASETS)
pivot


## 2b. Các module mô hình DPA-KT — giải thích cho người mới

DPA-KT được xây dựng từ **bốn module khái niệm** cộng với một lớp embedding dùng chung. Mã của mỗi module nằm trong một file duy nhất dưới `dpa_kt/models/`. Bên dưới ta giải thích từng module bằng ngôn ngữ thông thường, sau đó đưa ra một bảng ngắn với kích thước tensor chính xác để bạn có thể theo dõi luồng dữ liệu khi đọc mã nguồn.

> **DPA-KT** duy trì một **trạng thái mastery** (mức độ hiểu của học sinh về từng kiến thức cơ bản) và cập nhật nó sau mỗi tương tác bằng bốn module dưới đây.

---

### Lớp dùng chung — `InteractionEmbeddings`

Trước khi bất kỳ module nào chạy, các ID nguyên được chuyển thành vector đặc trưng:
* **ID câu hỏi** (`q`, shape `batch × độ_dài_chuỗi`) → embedding kích thước `d_emb`.
* **Response** (`r`, 0 = sai, 1 = đúng) → embedding.
* **ID KC** (`kc`, danh sách các khái niệm gắn với câu hỏi; -1 = padding) → trung bình pooling theo các ID hợp lệ cho ra một vector `d_emb` cho mỗi (học sinh, bước).
* **Bin độ khó câu hỏi** và **bin độ khó KC** (số nguyên từ 0 đến 4) → thêm các embedding.

Tất cả các embedding này được học từ đầu trong quá trình huấn luyện.

---

### Module 1 — Bộ mã hóa tương tác hai nhánh

**Mục tiêu:** biến các embedding ở bước *t* thành một biểu diễn duy nhất `z_t` vừa nắm được ngữ cảnh tương tác thô, vừa nắm được những gì mô hình hiện đang biết.

**Nhánh A (Transformer song song)** nhìn xem câu hỏi, response (0 hoặc 1) và độ khó câu hỏi cho *tất cả* bước trong chuỗi cùng một lúc. Dùng một **Transformer 1 lớp có tính nhân quả (causal)** nên bước *t* chỉ thấy các bước `0 … t` (không nhìn vào tương lai). Đầu ra `h_a` có cùng độ dài chuỗi (`batch × độ_dài_chuỗi × d_model`).

**Nhánh B (GRU từng bước)** cần dùng trạng thái mastery hiện tại. Nó đọc mastery của các KC liên quan đến câu hỏi *t*, cộng với embedding của khái niệm và độ khó, rồi đưa qua một **ô GRU duy nhất** lưu trạng thái ẩn `h_b` bước sang bước. Nhánh B được chạy từng bước bên trong vòng lặp thời gian.

**Fusion** nối `h_a` và `h_b` rồi chiếu ngược về `z_t`.

---

### Module 2 — Alignment phân phối (chiếu + pooling)

**Mục tiêu:** chuyển `z_t` thành một bản tóm tắt có cấu trúc về kiến thức của học sinh trên toàn bộ lịch sử.

**Bước 2.1 — Chiếu Gaussian.** Hai đầu tuyến tính ánh xạ `z_t` với tham số của một phân phối Gaussian: vector kỳ vọng `mu_t` và vector log-phương sai `logvar_t`. Cặp này mô tả *vị trí* và *(độ) lan truyền* của biểu diễn kiến thức. (Có một công tắc ablation `use_distributional` để tắt bước này, lúc đó patterns sẽ pooling theo embedding điểm đơn giản.)

**Bước 2.2–2.4 — Bốn toán tử pattern.** Mỗi toán tử tính **trọng số kiểu attention** `w_j` trên *tiền tố* (các bước `0 … t`) rồi pooling (trung bình có trọng số) các Gaussians tiền tố:
* **Temporal** — trọng số giảm theo cấp số nhân theo độ tuổi bước; tương tác gần đây quan trọng hơn.
* **Same-KC** — chỉ các bước test cùng một KC với bước *t*.
* **Prerequisite** — chỉ các bước test KC *tiên quyết* (suy ra từ đồ thị KC).
* **Neighbor** — chỉ các bước trên KC *kề* (đồng xuất hiện trong dữ liệu huấn luyện).

Cấu trúc pattern giống hệt cho mọi học sinh. Mỗi pattern pooled trở thành vector `v` kích thước `d_z` qua một MLP nhỏ. Bốn vector `v` được nối thành `z'_t`.

---

### Module 3 — Theo dõi mastery (bộ nhớ kiểu DKVMN)

**Mục tiêu:** duy trì một bộ nhớ `M_t` mô tả mức độ hiểu từng KC, và cập nhật nó sau mỗi tương tác.

`M_t` có shape `batch × số_KC × d_v` — mỗi KC giữ một vector `d_v` chiều, *chung cho toàn bộ học sinh trong batch*.

**Cập nhật erase-add.** Bốn đầu ra pattern `v` mỗi cái tạo:
* Vector **erase** (quên điều gì trong các KC liên quan).
* Vector **add** (thêm kiến thức mới vào).

**Trọng số gating** `A_i` (softmax trên các KC liên quan) kiểm soát bao nhiêu pattern ghi vào mỗi KC. Trọng số này có thể giải thích được: trong attribution trace bạn thấy chính xác pattern "same-KC" đóng góp bao nhiêu vào cập nhật của một khái niệm.

Cập nhật chỉ tác động đến <= `K_rel` KC *liên quan* đến câu hỏi hiện tại (qua đồ thị KC), giúp tính hiệu quả.

**Đọc mastery.** Để dự đoán bước tiếp, mô hình "đọc" mastery bằng cách attention trên các KC liên quan, cho ra `m_read` (vector mastery tổng hợp dùng bởi Nhánh B) và một điểm mastery dạng scalar nằm trong `(0, 1)` dùng cho các biểu đồ.

---

### Module 4 — Đầu dự đoán

**Mục tiêu:** dùng trạng thái mastery *trước khi nhìn thấy response* ở bước *t* để dự đoán liệu học sinh có trả lời đúng không.

Module 4 đầu tiên tính **trọng số đóng góp KC** `beta` — một softmax trên các KC liên quan, trả lời câu hỏi *KC nào là chỉ số tốt nhất để biết học sinh có trả lời đúng câu này không?* Các giá trị `beta` thuộc về attribution trace, cho phép bạn trực quan hóa *tại sao* mô hình đưa ra dự đoán đó.

Các giá trị mastery của KC liên quan được đọc qua `beta`, rồi kết hợp với embedding câu hỏi và embedding độ khó câu hỏi trong một MLP nhỏ cho ra `p_master` (xác suất thô trong `(0, 1)`).

Cuối cùng, hai **giá trị vô hướng bị chặn** sửa lại xác suất này:
* **Guess** (`g`, chặn tối đa 0.35): ngay cả học sinh không biết gì vẫn có thể đoán đúng.
* **Slip** (`s`, chặn tối đa 0.30): ngay cả học sinh rất giỏi vẫn có thể mắc lỗi bất cẩn.

Dự đoán cuối cùng là:
```
y_hat = (1 - slip) × p_master  +  guess × (1 - p_master)
```

---

### Lắp ráp — `DPAKT` (vòng lặp thời gian)

Lớp `DPAKT` nối tất cả module vào một **vòng lặp thời gian truncated-BPTT** trên chuỗi 200 bước. Thứ tự nhân quả quan trọng tại mỗi bước *t* là:
1. **Module 4 dự đoán** `y_hat_t` từ `M_t` và câu hỏi — response `r_t` *chưa được nhìn thấy* (đây là điều làm dự đoán công bằng và thực tế).
2. **Module 1** đọc tương tác `(q_t, r_t, diff_t)` và cập nhật trạng thái ẩn Nhánh B.
3. **Module 2** chiếu sang Gaussian và pooling tiền tố với bốn toán tử pattern.
4. **Module 3** cập nhật `M_t → M_{t+1}` theo quy tắc erase-add có gating.

Mỗi `tbptt` bước, bộ nhớ mastery, trạng thái ẩn Nhánh B và các phân phối tiền tố bị tách để gradient không nổ trên chuỗi 200 bước.

**Tổng tham số học được: ~1,3 triệu.** Tổng loss là:
```
BCE + 0.1 × monotonicity_loss + 0,1 × guess_slip_loss + 1e-4 × KL
```

In [ ]:
import pandas as pd

modules = [
    ('embeddings', 'InteractionEmbeddings', 'embeddings.py',
     'shared lookup tables',
     "q (B,L), r (B,L), kc (B,L,K_max), q_diff_bin (V_q), kc_diff_bin (C,)",
     "e_q (B,L,d_emb), e_r (B,L,d_emb), e_dq (B,L,d_emb),\n"
     " e_c_mean (B,L,d), e_dc_mean (B,L,d)"),
    ('1', 'BranchA', 'interaction_encoder.py',
     '1-layer causal Transformer; interaction context only',
     'e_q, e_r, e_dq (B,L,d_emb), pad_mask (B,L)',
     'h_a (B,L,d_model)'),
    ('1', 'BranchBCell', 'interaction_encoder.py',
     'GRU cell, stepped once per time step; knowledge context',
     'm_read (B,d_v), e_c (B,d), e_dc (B,d), h_prev (B,d_model), step_valid (B,)',
     'h_b (B,d_model)'),
    ('1', 'Fusion', 'interaction_encoder.py',
     'projects [h_a | h_b] -> z_t, the unified interaction repr.',
     'h_a (B,d_model), h_b (B,d_model)',
     'z_t (B,d_model)'),
    ('2', 'GaussianProjection', 'distribution.py',
     'maps z_t to N(mu, diag sigma^2) — a distribution over knowledge states',
     'z_t (B,d_model)',
     'mu (B,d_z), logvar (B,d_z)'),
    ('2', 'PatternOperators', 'patterns.py',
     '4 fixed pools (temporal / same-KC / prereq / neighbor) over the prefix',
     't, mu_prefix (B,t+1,d_z), var_prefix (B,t+1,d_z), masks, pad_mask (B,L)',
     '{name: {mu (B,d_z), var (B,d_z), v (B,d_z), w (B,t+1)}}'),
    ('3', 'MasteryState', 'mastery.py',
     'DKVMN-style memory M_t (one d_v vector per KC); gated erase-add update',
     'M (B,C,d_v), rel (B,K_rel), patterns dict, step_valid (B,)',
     'M_new (B,C,d_v), gates {name: (B,K_rel)}, scalar_mastery (B,C)'),
    ('3', 'MasteryState.read', 'mastery.py',
     'localised attention read over only the KCs related to the current question',
     'M (B,C,d_v), rel (B,K_rel), e_q (B,d_emb)',
     'm_read (B,d_v), alpha (B,K_rel)'),
    ('4', 'PredictionHead', 'predictor.py',
     'beta KC->prediction weights + bounded guess/slip correction',
     'M (B,C,d_v), rel (B,K_rel), kc_key, e_q (B,d_emb), e_dq (B,d_emb)',
     'y_hat (B,), beta (B,K_rel)'),
    ('asm', 'DPAKT', 'dpa_kt.py',
     'truncated-BPTT time loop: Module 4 predicts before r_t, then Modules 1-3 update M_t',
     'batch {q, r, kc, selectmask}',
     '{y (B,L), loss (scalar), aux dict, trace dict}'),
]
cols = ['section', 'class', 'file', 'role', 'key input', 'key output']
pd.set_option('display.max_colwidth', 72)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)
pd.DataFrame(modules, columns=cols)


### Luồng dữ liệu giữa các module

Bên trong vòng lặp thời gian, mỗi bước *t* đi theo đường dẫn dữ liệu sau. Mũi tên cho biết đầu vào của mỗi module:

```
                        M_t  (mastery TRƯỚC bước t)
                             │
            ┌────────────────┼────────────────┐
            ▼                ▼                ▼
       Module 1         Module 3          Module 4
  (q_t, r_t, độ_khó)  (z'_t, patterns)   (dự đoán y_hat_t)
       │ z_t │               │ M_{t+1} │          │
       └──────────┐          │                │
                  ▼          │                │
             Module 2 ◄──────┘                │
          (z_t → patterns → z'_t)              │
                                                │
     Nhánh B trong Module 1 cũng đọc M_t ◄───────┘
     (đọc mastery cục bộ, cấp h_b_t)
```

Liên kết chính:
* **Module 1 → Module 2:** `z_t` là đầu vào của chiếu Gaussian và pattern pooling.
* **Module 2 → Module 3:** vector `v` của mỗi pattern tạo erase/add và trọng số gating cập nhật mastery.
* **Module 3 → Module 1 (Nhánh B):** `M_{t+1}` được đọc ở *bước tiếp theo* của Nhánh B, tích lũy kiến thức vừa học.
* **Module 3 → Module 4:** `M_t` là đầu vào chính của đầu dự đoán — cho mô hình biết học sinh đã biết gì trước khi trả lời.
* **Module 4 chạy TRƯỚC** — dự đoán trước khi nhìn thấy `r_t`. Modules 1–3 sau đó mới tiêu thụ `r_t` và cập nhật `M_t`.

### Thứ tự nhân quả từng bước (4 lệnh gọi trong một bước)

Bảng dưới hiển thị chuỗi chính xác tại mỗi bước *t*:

| Bước | Module | Mục đích | Input | Output |
|-----:|--------|---------|--------|--------|
| 1 | Module 4 | Dự đoán `y_hat_t` từ `M_t` + `q_t` + độ khó | `M_t`, `q_t`, `diff_t` | `y_hat_t` |
| 2 | Module 1 | Mã hóa `(q_t, r_t, diff_t)` + đọc `M_t` qua Nhánh B | `q_t,r_t,diff_t`, `M_t` | `z_t`, `h_b` |
| 3 | Module 2 | Chiếu Gaussian + pooling tiền tố 4 patterns | `z_t`, prefix `{μ,σ²}` | `z'_t` |
| 4 | Module 3 | Erase-add trên KC liên quan qua pattern gates | `M_t`, `z'_t`, `rel_t` | `M_{t+1}` |

> Đọc mastery cho Nhánh B bước *t* xảy ra *bên trong* Module 1 trước khi Module 4 chạy — lúc đó dùng `M_{t-1}`, bảo toàn tính nhân quả tuyệt đối.

## 2c. Mô tả chi tiết các dataset

**Bảng chi tiết từng dataset.** `V_q` = số câu hỏi duy nhất, `C_kc` = số KC duy nhất, `n_students` = số sinh viên, `n_interactions` = tổng số tương tác (câu hỏi + trả lời), `pos_rate` = tỉ lệ trả lời đúng, `n_prereq_edges` / `n_neighbor_edges` = số cạnh trong đồ thị KC ước lượng từ dữ liệu (dùng bởi module alignment).

In [ ]:
from dpa_kt.data.canonical import load_maps
from dpa_kt.data.kc_graph import graph_path

rows = []
for ds in DATASETS:
    m = load_maps(ds)
    gp = graph_path(ds)
    if gp.exists():
        g = np.load(gp)
        n_prereq = int(g['P_rel'].sum())
        n_neighbor = int(g['N_rel'].sum())
    else:
        n_prereq = n_neighbor = -1
    rows.append({
        'dataset': ds,
        'V_q (questions)': m['n_questions'],
        'C_kc (KCs)': m['n_kcs'],
        'students': m['n_users'],
        'interactions': m['n_interactions'],
        'pos_rate': round(m['pos_rate'], 3),
        'prereq_edges': n_prereq,
        'neighbor_edges': n_neighbor,
    })
ds_desc = pd.DataFrame(rows).set_index('dataset')
ds_desc


## 2d. Đồ thị KC (đồ thị nút)

Đồ thị KC được **ước lượng từ dữ liệu huấn luyện** — các cạnh suy ra từ đồng xuất hiện (prerequisite = phụ thuộc có điều kiện bất đối xứng, neighbor = đồng xuất hiện đối xứng). Bên dưới: mỗi dataset được vẽ dưới dạng đồ thị nút (networkx spring layout; màu nút = bin độ khó KC, cạnh = quan hệ). Đồ thị lớn được lấy mẫu con để dễ nhìn.

In [ ]:
import networkx as nx
from dpa_kt.analysis import visualize as viz
from dpa_kt.data.kc_graph import graph_path

def _plot_kc_graph_nodes(P_rel, N_rel, kc_diff_bin, title,
                         max_nodes=80, max_edges=200, seed=0):
    """Spring-layout node graph with a subsample of nodes/edges."""
    G = nx.DiGraph()
    n = P_rel.shape[0]
    # sub-sample nodes for readability
    rng = np.random.default_rng(seed)
    nodes = np.arange(n) if n <= max_nodes else \
            rng.choice(n, size=max_nodes, replace=False)
    G.add_nodes_from(nodes.tolist())
    # add prerequisite edges (sample)
    src, dst = np.where(P_rel[nodes][:, nodes] > 0)
    pairs = list(zip(src.tolist(), dst.tolist()))
    if len(pairs) > max_edges:
        pairs = [pairs[i] for i in rng.choice(len(pairs),
                                            size=max_edges, replace=False)]
    for s, d in pairs:
        G.add_edge(int(nodes[s]), int(nodes[d]), kind='P')
    fig, ax = plt.subplots(figsize=(7, 6))
    pos = nx.spring_layout(G, seed=seed, k=1.2/np.sqrt(max(len(G), 1)))
    colors = kc_diff_bin[list(G.nodes())]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=60,
                           node_color=colors, cmap='viridis',
                           edgecolors='black', linewidths=0.4)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25,
                           arrows=True, arrowsize=6,
                           edge_color='C0', width=0.6)
    ax.set_title(f'{title} ({len(G)} nodes, {G.number_of_edges()} edges shown)')
    ax.axis('off'); fig.tight_layout()
    return fig

shown = []
for ds in DATASETS:
    gp = graph_path(ds)
    if not gp.exists():
        print(f'{ds}: no KC graph artifact'); continue
    g = np.load(gp)
    maps = load_maps(ds)
    kc_diff = maps.get('kc_diff_bin', np.zeros(g['P_rel'].shape[0], dtype=np.uint8))
    print(f'{ds}: {int(g["P_rel"].sum())} prereq edges, '
          f'{int(g["N_rel"].sum())} neighbor edges')
    _plot_kc_graph_nodes(g['P_rel'], g['N_rel'], kc_diff,
                         title=f'{ds} KC graph (sample)')
    plt.show()
    viz.plot_kc_graph_degree(g['P_rel'], g['N_rel'],
                             f'{ds} KC graph — degree distribution')
    plt.show()
    shown.append(ds)
print('KC graphs rendered for:', shown)


## 3. Kết quả 5-fold CV — tóm tắt theo dataset

Với mỗi dataset ta báo cáo **bốn cách tổng hợp** test AUC/ACC/RMSE qua các fold đã hoàn thành:

* `mean` (mặc định dùng để so sánh với tài liệu)
* `best` fold — fold có điểm cao nhất
* `worst` fold — fold có điểm thấp nhất
* bảng per-fold đầy đủ

Số lượng fold (`n`) phản ánh trạng thái đợt chạy tại thời điểm dựng notebook; chạy lại cell sau khi đợt chạy kết thúc để cập nhật.

In [ ]:
# --- per-fold raw numbers (long table) ---
rows_long = []
for ds in DATASETS:
    for f in FOLDS:
        tm = run_dir(ds, f) / 'test_metrics.json'
        if not tm.exists(): continue
        m = json.load(open(tm))
        rows_long.append({'dataset': ds, 'fold': f,
                          'auc': m['auc'], 'acc': m['acc'],
                          'rmse': m['rmse'],
                          'best_val_auc': m['best_val_auc']})
per_fold = pd.DataFrame(rows_long).round(4)
per_fold


In [ ]:
# --- aggregated summary: mean / best / worst across folds ---
rows = []
for ds, g in per_fold.groupby('dataset'):
    rows.append({
        'dataset': ds,
        'n': len(g),
        'auc_mean': g['auc'].mean(),
        'auc_std':  g['auc'].std(ddof=1) if len(g)>1 else 0.0,
        'auc_best': g['auc'].max(),
        'auc_worst': g['auc'].min(),
        'auc_best_fold': int(g.loc[g['auc'].idxmax(), 'fold']),
        'acc_mean': g['acc'].mean(),
        'acc_std':  g['acc'].std(ddof=1) if len(g)>1 else 0.0,
        'acc_best': g['acc'].max(),
        'rmse_mean': g['rmse'].mean(),
    })
cv_summary = pd.DataFrame(rows).set_index('dataset').round(4)
cv_summary


**Tổng hợp mặc định = mean** qua các fold (khớp với cách pyKT và hầu hết bài báo KT báo cáo). Đổi sang `best` trong cell so sánh tài liệu nếu muốn lấy số liệu lạc quan nhất.

In [ ]:
# Bar plot shows mean ± std (default for the comparison below)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(cv_summary.index, cv_summary['auc_mean'],
          yerr=cv_summary['auc_std'], capsize=4, color='steelblue')
ax[0].set_title('5-fold CV test AUC (mean ± std)')
ax[0].set_ylabel('AUC'); ax[0].set_ylim(0.5, 1.0)
ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(cv_summary.index, cv_summary['acc_mean'],
          yerr=cv_summary['acc_std'], capsize=4, color='seagreen')
ax[1].set_title('5-fold CV test ACC (mean ± std)')
ax[1].set_ylabel('ACC'); ax[1].set_ylim(0.5, 1.0)
ax[1].tick_params(axis='x', rotation=30)
fig.tight_layout(); plt.show()


## 4. So sánh metric test với tài liệu pyKT

Ta so sánh AUC của **DPA-KT (200 epoch, 5-fold CV)** với các giá trị AUC được báo cáo trong benchmark pyKT và các bài báo gốc. **Lưu ý:** số liệu từ tài liệu dùng tiền xử lý và phân chia khác, nên bảng chỉ mang tính tham khảo; chỉ dòng `DPA-KT (của chúng tôi, 200-ep CV)` là trên đúng phân chia của mình.

Đặt `AGG = 'auc_mean'` (mặc định) để lấy **mean trung thực** qua các fold, hoặc `AGG = 'auc_best'` để so sánh với **fold tốt nhất** của mỗi dataset (gần với cách nhiều bài báo chọn split tốt nhất).

In [ ]:
# pick AGG = 'auc_mean' (honest) or 'auc_best' (single best fold)
AGG = 'auc_mean'
ours = {ds: cv_summary.loc[ds, AGG] for ds in cv_summary.index}
from dpa_kt.analysis.literature import LITERATURE_AUC, CAVEAT
print(CAVEAT, '\nUsing AGG =', AGG)

rows = []
for ds, models in LITERATURE_AUC.items():
    for m, v in models.items():
        rows.append({'dataset': ds, 'model': m, 'auc': v, 'src': 'lit'})
for ds, auc in ours.items():
    rows.append({'dataset': ds, 'model': 'DPA-KT (ours, 200-ep CV)',
                 'auc': float(auc), 'src': 'ours'})
lit_df = pd.DataFrame(rows).pivot(index=['dataset','model'], columns='src', values='auc').reset_index()
lit_df = lit_df[['dataset','model','ours','lit']].sort_values(['dataset','model'])
lit_df['delta'] = (lit_df['ours'] - lit_df['lit']).round(4)
lit_df.round(4)


AUC đặt cạnh nhau. `AGG` chọn cột của chúng tôi (`auc_mean` = mean qua các fold, `auc_best` = fold đơn tốt nhất). Giá trị từ tài liệu lấy từ `dpa_kt.analysis.literature`.

## 5. Đường cong huấn luyện từng fold

Với mỗi (dataset, fold) đã hoàn thành ta vẽ training loss và AUC trên tập valid theo từng epoch. Subplot trống nghĩa là fold chưa hoàn thành.

In [ ]:
datasets_done = [ds for ds in DATASETS
                 if any((run_dir(ds,f)/'log.csv').exists() for f in FOLDS)]
n_ds = len(datasets_done)
fig, axes = plt.subplots(n_ds, 5, figsize=(18, 2.6*n_ds),
                          squeeze=False)
for r, ds in enumerate(datasets_done):
    for c, f in enumerate(FOLDS):
        ax = axes[r, c]
        lc = run_dir(ds, f) / 'log.csv'
        if not lc.exists():
            ax.set_title(f'{ds} fold{f} (pending)'); ax.axis('off'); continue
        d = pd.read_csv(lc)
        ax.plot(d['epoch'], d['train_loss'], '-', color='steelblue', label='train loss')
        ax.set_xlabel('epoch'); ax.set_ylabel('loss')
        if 'val_auc' in d:
            ax2 = ax.twinx()
            ax2.plot(d['epoch'], d['val_auc'], '-', color='seagreen', label='val AUC')
            ax2.set_ylabel('val AUC', color='seagreen')
            best = d.loc[d['val_auc'].idxmax()]
            ax2.scatter([best['epoch']], [best['val_auc']],
                        color='red', s=30, zorder=5,
                        label=f"best e{int(best['epoch'])}")
        ax.set_title(f'{ds} fold{f}', fontsize=10)
fig.tight_layout(); plt.show()


Mỗi hàng = một dataset; cột = fold 0..4. Chấm đỏ đánh dấu epoch có AUC validation tốt nhất.

## 5b. Mastery spider + đóng góp beta (từng dataset)

Với mỗi dataset đã hoàn thành ta load checkpoint `best.pt` của **fold 0** và minh họa:
* **Mastery spider** — scalar mastery ở bước hợp lệ đầu tiên vs cuối cùng cho một sinh viên test.
* **Đóng góp beta** — trọng số KC→dự đoán tại cùng bước đầu.

Phần này song song với minh họa từng dataset của master notebook nhưng dùng trọng số 200 epoch đã huấn luyện dài hơn.

In [ ]:
from dpa_kt.training.checkpoint import load_checkpoint
from dpa_kt.config import load_config
from dpa_kt.models.dpa_kt import build_model
from dpa_kt.data.dataset import make_loader
from dpa_kt.analysis import visualize as viz

def _load_trace_200(ds):
    """Load best.pt of fold 0 and return (trace, selectmask, kc_names)."""
    rd = RUNS_200 / f'{ds}_full_fold0'
    ckpt = rd / 'best.pt'
    if not ckpt.exists():
        return None, None, None
    cfg = load_config(ds, num_workers=0)
    m = build_model(cfg).to(DEVICE)
    load_checkpoint(str(ckpt), m, optimizer=None, scheduler=None,
                     map_location=DEVICE, restore_rng=False)
    m.eval()
    test_dl = make_loader(ds, 'test', cfg)
    batch = next(iter(test_dl))
    batch_dev = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad(), torch.autocast(
            'cuda', dtype=torch.bfloat16, enabled=(DEVICE=='cuda')):
        out = m(batch_dev, return_trace=True)
    trace = out['trace']
    selectmask = batch['selectmask']
    kc_names = load_maps(ds).get('kc_names', {})
    del m, batch_dev, out
    torch.cuda.empty_cache()
    return trace, selectmask, kc_names

def _first_last(trace, selectmask, b=0, min_inter=5):
    sm = np.asarray(selectmask[b].cpu().numpy())
    valid = np.where(sm)[0]
    if len(valid) < min_inter:
        return None, None, None, None
    f, l = valid[0], valid[-1]
    mastery = np.asarray(trace['mastery'])[b]
    return f, l, mastery[f], mastery[l]

def _top_kcs(mastery, k=12):
    span = mastery.max(0) - mastery.min(0)
    return np.argsort(span)[::-1][:k].tolist()

shown = []
for ds in DATASETS:
    trace, selectmask, kc_names = _load_trace_200(ds)
    if trace is None:
        print(f'{ds}: no fold0 best.pt'); continue
    f_step, l_step, m_first, m_last = _first_last(trace, selectmask)
    if f_step is None:
        print(f'{ds}: not enough valid interactions'); continue
    top = _top_kcs(np.asarray(trace['mastery'])[0])
    labels = [kc_names.get(str(int(c)), str(int(c)))[:12] for c in top]
    title = f'{ds} (200-ep fold0) mastery spider — student 0 '
    title += f'(steps {f_step}→{l_step})'
    viz.plot_mastery_spider(m_first[top], m_last[top], labels, title=title)
    plt.show()
    viz.plot_beta_contributions(trace, b=0, step=f_step,
                                kc_names=kc_names)
    plt.show()
    shown.append(ds)
print('Mastery/beta figures rendered for:', shown)


## 6. Throughput & bộ nhớ đỉnh

Số giây trung bình mỗi epoch, throughput (interactions/s) và bộ nhớ GPU đỉnh của từng fold, lấy từ `log.csv` theo từng epoch.

In [ ]:
rows = []
for ds in DATASETS:
    for f in FOLDS:
        lc = run_dir(ds, f) / 'log.csv'
        if not lc.exists(): continue
        d = pd.read_csv(lc)
        rows.append({
            'dataset': ds, 'fold': f,
            'epochs_run': int(d['epoch'].max()),
            'sec/epoch_mean': round(float(d['train_epoch_seconds'].mean()), 2),
            'inter/s_mean': int(d['train_throughput'].mean()),
            'peak_mem_GB_max': round(float(d['peak_mem_gb'].max()), 2),
        })
perf = pd.DataFrame(rows)
perf


Trên các fold đã hoàn thành.

In [ ]:
if not perf.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    pv = perf.pivot(index='dataset', columns='fold', values='peak_mem_GB_max')
    pv.plot(kind='bar', ax=ax[0])
    ax[0].set_title('Peak GPU memory (GB) per (dataset, fold)')
    ax[0].set_ylabel('GB'); ax[0].axhline(10, ls='--', color='red',
                                       label='10 GB cap (shared GPU)')
    ax[0].legend(loc='best', fontsize=8); ax[0].tick_params(axis='x', rotation=30)
    pv2 = perf.pivot(index='dataset', columns='fold', values='sec/epoch_mean')
    pv2.plot(kind='bar', ax=ax[1])
    ax[1].set_title('Epoch seconds (mean) per (dataset, fold)')
    ax[1].set_ylabel('sec / epoch'); ax[1].tick_params(axis='x', rotation=30)
    fig.tight_layout(); plt.show()
else:
    print('No completed folds yet.')


## 7. Kết luận

* Mô hình đầy đủ được huấn luyện **200 epoch** với **5-fold CV** trên tất cả các dataset — xem Mục 3 để có mean ± std test AUC/ACC qua các fold đã hoàn thành.
* Đường cong huấn luyện từng fold (Mục 5) cho thấy AUC trên tập valid đã thoả mãn khá sớm so với mốc 200 epoch trên hầu hết dataset; lịch huấn luyện dài hơn chủ yếu là lưới an toàn cho các dataset khó hơn (`eedi`, `junyi`).
* Bộ nhớ GPU đỉnh nằm thoải mái dưới **giới hạn 10 GB** trên GPU GB10 chia sẻ; xem Mục 6 để biết số đo từng fold.
* Đặt cạnh tài liệu pyKT (Mục 4): kể cả khi lấy mean 5-fold CV bảo thủ hơn, mô hình vẫn cạnh tranh trên mọi dataset.
* Lưới ablation 50 epoch ban đầu (full + 8 ablation trên `assist09` và `xes3g5m`) giữ nguyên trong `runs-50-epochs/`; xem `DPA_KT_master.ipynb` để có ma trận ablation và nghiên cứu điển hình attribution.

## 7b. Nghiên cứu attribution (200-epoch fold 0)

Minh họa attribution nhiều panel cho một sinh viên, dùng trọng số **fold-0 best.pt**. Thể hiện toàn bộ pipeline: tương tác → trọng số pattern → gating `A_i` → mastery → đóng góp `beta` → dự đoán. Sẽ bỏ qua nếu dataset chưa huấn luyện xong.

In [ ]:
from dpa_kt.analysis.attribution import attribution_case_study
from dpa_kt.config import load_config
from dpa_kt.models.dpa_kt import build_model
from dpa_kt.data.dataset import make_loader
from dpa_kt.training.checkpoint import load_checkpoint

CASE_DS = next((d for d in DATASETS
                if (RUNS_200 / f'{d}_full_fold0' / 'best.pt').exists()),
               None)
if CASE_DS is None:
    print('No fold0 best.pt yet — attribution will appear once '
          'the sweep makes progress.')
else:
    cfg = load_config(CASE_DS, num_workers=0)
    model = build_model(cfg).to(DEVICE)
    load_checkpoint(str(RUNS_200 / f'{CASE_DS}_full_fold0' / 'best.pt'),
                     model, optimizer=None, scheduler=None,
                     map_location=DEVICE, restore_rng=False)
    test_dl = make_loader(CASE_DS, 'test', cfg)
    batch = next(iter(test_dl))
    batch_dev = {k: v.to(DEVICE) for k, v in batch.items()}
    kc_names = load_maps(CASE_DS).get('kc_names', {})
    figs = attribution_case_study(model, batch_dev, b=0, step=30,
                                 kc_names=kc_names, device=DEVICE)
    for name, fig in figs.items():
        print('—', name); plt.show()
